In [33]:
import polars as pl
import numpy as np
from functools import reduce

In [2]:
# load data
df_cleaned = pl.read_parquet("data/cleaned.parquet")

## status_ratios

In [35]:
df_status_ratio = df_cleaned.with_columns(
    (pl.col("http.response.status_code").str.slice(0, 1) + "xx")
    .alias("status_group")
)

counts = (
    df_status_ratio
    .group_by(["source.ip", "status_group"])
    .len()
    .rename({"len": "count"})
)

ratios = counts.with_columns(
    (pl.col("count") / pl.col("count").sum().over("source.ip"))
    .alias("ratio")
)

ratios_wide = (
    ratios
    .pivot(
        index="source.ip",
        on="status_group",
        values="ratio"
    )
    .fill_null(0)
)

ratios_wide.head(5)

source.ip,3xx,2xx,5xx,4xx
str,f64,f64,f64,f64
"""143.244.57.242""",1.0,0.0,0.0,0.0
"""80.110.8.182""",1.0,0.0,0.0,0.0
"""46.101.169.193""",0.6,0.4,0.0,0.0
"""1.48.51.149""",1.0,0.0,0.0,0.0
"""177.11.92.78""",1.0,0.0,0.0,0.0


## active_days_ratio

In [4]:
# Step 1: extract the date
df_days = df_cleaned.with_columns(
    pl.col("timestamp").dt.date().alias("date")
)

# Step 2: compute total number of unique days in dataset
total_days = df_days.select(pl.col("date").n_unique()).item()

# Step 3: count distinct days per IP
df_days_ratio = (
    df_days
    .group_by("source.ip")
    .agg(
        pl.n_unique("date").alias("active_days")
    )
    .with_columns(
        (pl.col("active_days") / total_days).alias("active_days_ratio")
    )
    .sort("active_days_ratio", descending=True)
).select(
    "source.ip", "active_days_ratio"
)

df_days_ratio.head(5)


source.ip,active_days_ratio
str,f64
"""85.158.206.20""",1.0
"""37.46.140.139""",1.0
"""37.46.139.102""",1.0
"""114.119.129.211""",1.0
"""35.240.105.55""",1.0


## request_count

In [37]:
df_request_count = df_cleaned.group_by("source.ip").agg(
    pl.len().alias("request_count")
    ).sort("request_count", descending=True)
df_request_count.head(5)

source.ip,request_count
str,u32
"""172.18.0.2""",202436
"""35.205.61.42""",40348
"""37.46.140.139""",40344
"""176.9.136.188""",40328
"""109.235.78.11""",40327


## endpoint_entropy

In [6]:
# Step 1: count requests per endpoint per IP
counts = (
    df_cleaned.group_by(["source.ip", "url.path"])
    .agg(pl.len().alias("count"))
)

# Step 2: compute probabilities
counts = counts.with_columns(
    (pl.col("count") / pl.col("count").sum().over("source.ip")).alias("p")
)

# # Step 3: compute entropy per IP
entropy_per_ip = counts.group_by("source.ip").agg(
    (-1 * (pl.col("p") * np.log2(pl.col("p")))).sum().alias("endpoint_entropy")
)


entropy_per_ip.sort("endpoint_entropy", descending=True).head(5)


source.ip,endpoint_entropy
str,f64
"""116.203.157.62""",7.976097
"""114.119.129.211""",7.539335
"""114.119.132.211""",7.490418
"""114.119.154.11""",7.453946
"""114.119.146.7""",7.430829


## time_entropy

In [7]:
# Step 1: extract hour/minute bin (or 5-min bin)
df_time = df_cleaned.with_columns(
    pl.col("timestamp").dt.truncate("5m").alias("time_bin")
)

# Step 2: count requests per IP per time bin
counts = (
    df_time.group_by(["source.ip", "time_bin"])
    .agg(pl.len().alias("count"))
)

# Step 3: compute probability of each time bin for each IP
counts = counts.with_columns(
    (pl.col("count") / pl.col("count").sum().over("source.ip")).alias("p")
)

# Step 4: compute time entropy per IP (Shannon entropy)
time_entropy = counts.group_by("source.ip").agg(
    (-(pl.col("p") * np.log2(pl.col("p")))).sum().alias("time_entropy")
)

time_entropy.sort("time_entropy", descending=True).head(5)


source.ip,time_entropy
str,f64
"""37.46.139.102""",10.977073
"""37.46.139.46""",10.977054
"""109.235.78.11""",10.977022
"""176.9.136.188""",10.976979
"""37.46.140.139""",10.976979


## geo_entropy

In [27]:
df_cleaned["source.geo"][0]

{'location': {'lat': 52.6318, 'lon': 4.7409},
 'city_name': 'Alkmaar',
 'country_name': 'Netherlands',
 'country_iso_code': 'NL'}

In [32]:
geo_iso_code = df_cleaned.select(
    pl.col("source.ip"),
    pl.col("source.geo")
      .struct.field("country_iso_code")
      .fill_null("unknown")
      .alias("country_iso_code")
)

geo_counts = (
    geo_iso_code.group_by(["source.ip", "country_iso_code"])
    .agg(pl.len().alias("count"))
)

ips_multi_country = (
    geo_counts
    .group_by("source.ip")
    .agg(
        pl.col("country_iso_code").n_unique().alias("n_countries")
    )
    .filter(pl.col("n_countries") > 1)
    .sort("n_countries", descending=True)
)
ips_multi_country

source.ip,n_countries
str,u32


### merging all features

In [40]:
dfs = [
    ratios_wide,
    df_days_ratio,
    df_request_count,
    entropy_per_ip,
    time_entropy
]

df_features = reduce(
    lambda left, right: left.join(
        right,
        on="source.ip",
        how="full",
        coalesce=True
    ),
    dfs
)

In [41]:
df_features.head(5)

source.ip,3xx,2xx,5xx,4xx,active_days_ratio,request_count,endpoint_entropy,time_entropy
str,f64,f64,f64,f64,f64,u32,f64,f64
"""172.105.89.161""",1.0,0.0,0.0,0.0,0.714286,6,-0.0,2.584963
"""186.32.219.176""",1.0,0.0,0.0,0.0,0.142857,1,-0.0,-0.0
"""1.180.246.82""",1.0,0.0,0.0,0.0,0.142857,1,-0.0,-0.0
"""185.191.171.44""",0.0,1.0,0.0,0.0,0.142857,1,-0.0,-0.0
"""185.191.171.40""",0.0,1.0,0.0,0.0,0.142857,1,-0.0,-0.0


In [42]:
df_features.write_parquet("data/features.parquet")